In [18]:
import pandas as pd
import numpy as np

In [19]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_CRRI Mathura Road, Delhi - IMD.xlsx",skiprows=16)

In [20]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,BP,Xylene,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,169.03,298.75,104.05,36.01,97.64,1.45,17.18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,02-01-2025 00:00,03-01-2025 00:00,158.17,268.48,73.42,32.37,70.20,1.65,19.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,03-01-2025 00:00,04-01-2025 00:00,264.44,475.04,124.69,53.86,143.41,2.05,27.98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,04-01-2025 00:00,05-01-2025 00:00,227.10,342.68,64.47,39.67,87.30,3.05,30.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,05-01-2025 00:00,06-01-2025 00:00,139.23,236.75,11.08,31.93,24.66,1.14,22.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,361.59,506.98,94.03,49.74,102.91,1.76,39.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
316,13-11-2025 00:00,14-11-2025 00:00,308.69,435.04,65.99,54.46,82.62,1.36,38.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
317,14-11-2025 00:00,15-11-2025 00:00,208.67,361.67,52.21,26.01,56.28,0.94,32.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
318,15-11-2025 00:00,16-11-2025 00:00,214.53,391.52,62.42,35.52,69.63,1.30,37.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [21]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 11)


In [22]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
TOT-RF       0
dtype: int64


In [23]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [ ]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [24]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 10)
          From Date           To Date   PM2.5    PM10      NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  169.03  298.75  104.05  36.01   97.64   
1  02-01-2025 00:00  03-01-2025 00:00  158.17  268.48   73.42  32.37   70.20   
2  03-01-2025 00:00  04-01-2025 00:00   58.33  186.89  124.69  53.86  143.41   
3  04-01-2025 00:00  05-01-2025 00:00   58.33  342.68   64.47  39.67   87.30   
4  05-01-2025 00:00  06-01-2025 00:00  139.23  236.75   11.08  31.93   24.66   

     CO  Ozone  TOT-RF  
0  1.45  17.18       0  
1  1.65  19.54       0  
2  2.05  27.98       0  
3  3.05  30.54       0  
4  1.14  22.23       0  


In [25]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [26]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,2.489548,1.119920,0.956295,-0.055513,0.628276,0.209856,-1.463443,0.0
1,02-01-2025 00:00,03-01-2025 00:00,2.229521,0.800965,0.316983,-0.214972,0.062106,0.452683,-1.324832,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.161006,-0.058749,1.387095,0.726449,1.572649,0.938337,-0.829121,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.161006,1.582810,0.130178,0.104822,0.414931,2.152473,-0.678764,0.0
4,05-01-2025 00:00,06-01-2025 00:00,1.776029,0.466626,-0.984183,-0.234247,-0.877522,-0.166526,-1.166839,0.0
...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.161006,-0.058749,0.747157,0.545963,0.737012,0.586238,-0.178354,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.161006,2.556007,0.161903,0.752733,0.318368,0.100584,-0.217118,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.161006,1.782908,-0.125714,-0.493587,-0.225106,-0.409353,-0.577742,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.161006,2.097437,0.087390,-0.076978,0.050345,0.027735,-0.265867,0.0


In [27]:
df.to_excel('CRRIMathura2025.xlsx', index=False)